In [6]:
# Mount Google Drive

from google.colab import drive
drive.mount('/content/gdrive')

Mounted at /content/gdrive


In [7]:
# Prepare data

!scp '/content/gdrive/My Drive/ColabNotebooks/Mini-Project-3/data.zip' '/content/data.zip'

!unzip '/content/data.zip' -d '/content/'

Archive:  /content/data.zip
   creating: /content/data/
  inflating: /content/data/a.webp    
  inflating: /content/data/b.png     
  inflating: /content/data/c.jpg     
  inflating: /content/data/d.jpg     


In [8]:
# Install

!apt install tesseract-ocr
!apt install libtesseract-drive

!pip install pytesseract
!pip install Pillow
!pip install easyocr
!pip install boto3

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
tesseract-ocr is already the newest version (4.1.1-2.1build1).
0 upgraded, 0 newly installed, 0 to remove and 6 not upgraded.
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
E: Unable to locate package libtesseract-drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 68.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 978.2/978.2 kB 45.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 300.6/300.6 kB 23.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.6/140.6 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.7/14.7 MB 63.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 5.2 MB/s eta 0:00:00


In [19]:
# Text Detection

import pytesseract
from PIL import Image
from easyocr import Reader

reader = Reader(['en'])

def read_text_tesseract(image_path):
  text = pytesseract.image_to_string(Image.open(image_path), lang='eng')
  return text

def read_text_easyocr(image_path):
  text = ''
  results = reader.readtext(Image.open(image_path))
  for result in results:
    text += result[1] + ' '
  text = text[:-1]
  return text

In [31]:
import re
import os

def jaccard_similarity(sentence1, sentence2):
    # Normalize: lowercase and remove punctuation
    def tokenize(text):
        text = text.lower()
        text = re.sub(r'[^\w\s]', '', text)  # remove punctuation
        return set(text.split())

    set1 = tokenize(sentence1)
    set2 = tokenize(sentence2)

    # Handle edge case where both sets are empty
    if not set1 and not set2:
        return 1.0

    intersection = set1.intersection(set2)
    union = set1.union(set2)

    return len(intersection) / len(union)

score_tesseract = 0
score_easyocr = 0

for image_path_ in os.listdir('/content/data'):
  image_path= os.path.join('/content/data', image_path_)
  print(image_path)

  gt = image_path[:-4].replace('_', ' ').lower()

  score_tesseract += jaccard_similarity(gt, read_text_tesseract(image_path).lower().replace('\n', '').replace('!', '').replace('?', '').replace(' ', ''))
  score_easyocr += jaccard_similarity(gt, read_text_easyocr(image_path).lower().replace('\n', '').replace('!', '').replace('?', '').replace(' ', ''))

  break

print('Tesseract: ', score_tesseract/2)
print('Easyocr: ', score_easyocr/2)

/content/data/c.jpg
Tesseract:  0.0
Easyocr:  0.0


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
